# Market Sentiment Analysis of Guardian Business Articles

## Overview
This project investigates whether machine learning models can reliably classify 
the sentiment of financial news articles from *The Guardian* on the scale below:
- 4 - POSITIVE
- 3 - LEANING POSITIVE
- 2 - NEUTRAL
- 1 - LEANING NEGATIVE
- 0 - NEGATIVE

## Models
We used four models over the course of this project evaluated on ground truth labels generated by ensembling two LLMs — **ZotGPT** and **Gemini** — via their respective APIs, averaging their predictions per article.

- Logistic Regression + TF-IDF - Baseline
- FinBERT (zero-shot) - Pretrained transformer baseline 
- FinBERT (fine-tuned) - Using hugging-face transformers
- LLaMA Zero-Shot and Few-Shot Baseline Method

This notebook will go over a simple pipeline on a sample dataset of articles using the baseline FinBert model as inference, including data proprocessing, model evaluation, and stock market correlation.


In [9]:
import json 
import os 
import pandas as pd 
from transformers import pipeline
from sklearn.metrics import mean_absolute_error, accuracy_score, classification_report
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

## LLM Committee Labeling

Each article's title and first 2,000 characters of body text were extracted and formatted into a structured prompt for LLM labeling. Let's take a look at what a prompt that goes into the API call might look like.

In [2]:
with open("../data/raw/sample_guardian_articles.json", encoding="utf-8") as f:
    articles = json.load(f)
    
def create_sentiment_prompt(article):
    title = article.get("webTitle", "")
    body = article.get("fields", {}).get("bodyText", "")

    text_to_analyze = f"{title}\n\n{body[:2000]}"

    prompt = f"""You are a financial sentiment analysis expert. Analyze the following Guardian business article and classify its sentiment regarding the US stock market.

Article:
{text_to_analyze}

Classify the sentiment into one of these categories:
4 - POSITIVE: Optimistic outlook, growth, gains, bullish sentiment, or favorable conditions for US stocks
3 - LEANING POSITIVE: Mostly positive but with some caveats or qualifiers
2 - NEUTRAL: Balanced view, mixed signals, or not directly related to US stock market sentiment
1 - LEANING NEGATIVE: Mostly negative but with some caveats or qualifiers
0 - NEGATIVE: Pessimistic outlook, losses, bearish sentiment, concerns, or unfavorable conditions for US stocks

Respond with ONLY the number (0, 1, 2, 3, or 4) and nothing else."""

    return prompt

print(create_sentiment_prompt(articles[0]))

You are a financial sentiment analysis expert. Analyze the following Guardian business article and classify its sentiment regarding the US stock market.

Article:
NatWest is chasing the mass affluent wallet. So is everyone else | Nils Pratley

Announce a £2.7bn acquisition and watch your stock market value fall by £3.1bn. NatWest picked a bad day to announce its big move in the fashionable field of “wealth management” – the noise from Westminster created a poor backdrop for UK assets such as gilts and domestic banks. But the main problem with its Evelyn Partners deal is that it is very much of the “one for the long term” variety. The terms are not obviously cheap. The NatWest chief executive Paul Thwaite’s strategic thinking wasn’t the controversial element because it is conventional. Lloyds, HSBC and Barclays are all talking about the appeal of managing money for “the mass affluent”, a cohort that tends to mean those with at least £50,000 to invest, but in practice usually implies muc

## Building the Dataset

The output labels from the prompt input into ZotGPT and Gemini were then averaged and stored in new JSON object. From here, we'll create a label map mapping each label to an article ID. This dictionary was then used to map an article's important text to its id, producing a parallel texts and labeled list. We will then have two matching arrays of texts and labels ready for training.

In [14]:
def load_averaged_labels(path):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    label_map = {}
    for id, score in data.items():
        label_map[id] = int(score)

    return label_map

def build_dataset_from_labels(articles, label_map):
    texts = []
    labels = []

    for art in articles:
        headline = art["fields"]["headline"]
        standfirst = art["fields"]["standfirst"]
        body = art["fields"]["bodyText"]

        text = headline + " " + standfirst + " " + body

        texts.append(text)
        labels.append(label_map[art["id"]])

    return texts, labels

averaged_labels = load_averaged_labels("../data/processed/sample_averaged_labels.json")
texts_eval, y_eval = build_dataset_from_labels(articles, averaged_labels)


## FinBERT Baseline (Zero-Shot)

This zero-shot FinBERT will serve as a strong pretrained baseline. Since FinBERT natively outputs three classes (positive,
neutral, negative), predictions were mapped to our 0–4 scale using confidence thresholding:

In [17]:
def run_finbert(texts):
    finbert = pipeline(
        "text-classification",
        model="ProsusAI/finbert"
    )
    y_pred = []
    for text in texts:
        result = finbert(
            text,
            truncation=True,
            max_length=512
        )[0]

        label = result["label"].lower()
        score = result["score"]

        if label == "negative":
            if score > 0.75:
                y_pred.append(0)
            else:
                y_pred.append(1)

        elif label == "neutral":
            y_pred.append(2)

        elif label == "positive":
            if score > 0.75:
                y_pred.append(4)
            else:
                y_pred.append(3)

    return y_pred

In [18]:
def evaluate_model(y_true, y_pred):
    """ 
    Runs all evaluation metrics and prints results.
    """

    mae = mean_absolute_error(y_true, y_pred)
    acc = accuracy_score(y_true, y_pred)

    print("\n" + "=" * 60)
    print("=" * 60)

    print(f"Accuracy: {acc:.3f}")
    print(f"MAE: {mae:.3f}")

    print("\nClassification Report:")
    print(classification_report(y_true, y_pred))

evaluate_model(y_eval, run_finbert(texts_eval))

[2, 2, 4, 2, 0, 3, 4, 4, 3, 0, 2, 0, 0, 0, 0, 0, 0, 2, 4, 2]


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 564.49it/s, Materializing param=classifier.weight]                                      
BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Accuracy: 0.600
MAE: 0.750

Classification Report:
              precision    recall  f1-score   support

           0       0.70      0.88      0.78         8
           1       0.00      0.00      0.00         0
           2       0.50      0.33      0.40         6
           3       1.00      0.50      0.67         2
           4       0.67      0.50      0.57         4

    accuracy                           0.60        20
   macro avg       0.57      0.44      0.48        20
weighted avg       0.66      0.60      0.61        20



c:\Users\robbi\source\repos\CS175-Market-Sentiment\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\robbi\source\repos\CS175-Market-Sentiment\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\robbi\source\repos\CS175-Market-Sentiment\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"